# ЛР 4.1. Трансфер обучения и Fine-tuning на ResNet

Предобученная ResNet18 → feature extraction → fine-tuning → сравнение с обучением с нуля. Датасет: Oxford Flowers 102.


### Цель

- Освоить Transfer Learning и Fine-tuning
- Заменить голову классификации
- Сравнить freeze / fine-tune / from-scratch


In [ ]:
# !pip install torch torchvision matplotlib scikit-learn pillow

import time
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 1. Данные и аугментации

Нормализация должна соответствовать ImageNet.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 102
EPOCHS_HEAD = 5
EPOCHS_FT = 5
EPOCHS_SCRATCH = 5
LR_HEAD = 1e-3
LR_FT = 1e-4
DATA_ROOT = './flowers102'

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# Flowers102: splits 'train' / 'val' / 'test'
train_ds = datasets.Flowers102(DATA_ROOT, split='train', download=True, transform=train_tf)
val_ds = datasets.Flowers102(DATA_ROOT, split='val', download=True, transform=eval_tf)
test_ds = datasets.Flowers102(DATA_ROOT, split='test', download=True, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(len(train_ds), len(val_ds), len(test_ds))


## 2. Утилиты обучения


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return running_loss / total, correct / total


def fit(model, optimizer, epochs, tag='model'):
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_wts, best_acc = copy.deepcopy(model.state_dict()), 0.0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        if va_acc > best_acc:
            best_acc = va_acc
            best_wts = copy.deepcopy(model.state_dict())
        print(f'[{tag}] epoch {epoch}/{epochs}  train_acc={tr_acc:.3f}  val_acc={va_acc:.3f}')
    model.load_state_dict(best_wts)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f'[{tag}] test_acc={test_acc:.4f}  time={time.time()-t0:.1f}s')
    return history, test_acc


def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history['train_acc'], label='train')
    ax[0].plot(history['val_acc'], label='val')
    ax[0].set_title(f'{title} accuracy'); ax[0].legend()
    ax[1].plot(history['train_loss'], label='train')
    ax[1].plot(history['val_loss'], label='val')
    ax[1].set_title(f'{title} loss'); ax[1].legend()
    plt.tight_layout(); plt.show()


## 3. Feature extraction (замороженный backbone)


In [ ]:
def build_resnet(pretrained=True, freeze_backbone=True):
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)
    return model.to(device)

model_fe = build_resnet(pretrained=True, freeze_backbone=True)
# обучаем только голову
params_fe = [p for p in model_fe.parameters() if p.requires_grad]
opt_fe = optim.Adam(params_fe, lr=LR_HEAD)
hist_fe, acc_fe = fit(model_fe, opt_fe, EPOCHS_HEAD, tag='feature_extraction')
plot_history(hist_fe, 'Feature extraction')


## 4. Fine-tuning

Разморозьте часть слоёв и уменьшите LR.


In [ ]:
model_ft = copy.deepcopy(model_fe)

# Размораживаем layer3, layer4 и fc (пример)
for name, p in model_ft.named_parameters():
    if name.startswith('layer3') or name.startswith('layer4') or name.startswith('fc'):
        p.requires_grad = True
    else:
        p.requires_grad = False

params_ft = [p for p in model_ft.parameters() if p.requires_grad]
opt_ft = optim.Adam(params_ft, lr=LR_FT)
hist_ft, acc_ft = fit(model_ft, opt_ft, EPOCHS_FT, tag='fine_tuning')
plot_history(hist_ft, 'Fine-tuning')


## 5. Обучение с нуля


In [ ]:
model_sc = build_resnet(pretrained=False, freeze_backbone=False)
opt_sc = optim.Adam(model_sc.parameters(), lr=LR_HEAD)
hist_sc, acc_sc = fit(model_sc, opt_sc, EPOCHS_SCRATCH, tag='from_scratch')
plot_history(hist_sc, 'From scratch')


## 6. Сравнение результатов


In [ ]:
results = {
    'Feature extraction': acc_fe,
    'Fine-tuning': acc_ft,
    'From scratch': acc_sc,
}
print('Test accuracy:')
for k, v in results.items():
    print(f'  {k:20s} {v:.4f}')

# ЗАДАНИЕ: кратко прокомментируйте таблицу в Markdown-ячейке ниже.


**Вопрос 1.** Почему для предобученной ResNet нужна ImageNet-нормализация?

**Ответ:**

---

**Вопрос 2.** Какие слои обучаются на этапе feature extraction и почему это ускоряет обучение?

**Ответ:**

---

**Вопрос 3.** Почему при fine-tuning learning rate обычно уменьшают?

**Ответ:**
